# Visualização de Dados FHIR - Mini HAPI

Este notebook demonstra como **consultar e visualizar dados FHIR** do servidor Mini HAPI.

## Fluxo Completo do TCC

```
┌─────────────────┐       ┌──────────────────┐       ┌─────────────────┐       ┌──────────────────┐
│  Sistema Legado │  -->  │  Conector ETL    │  -->  │   Mini HAPI     │  -->  │  Visualização    │
│  (Simulador)    │       │  (Transformação) │       │   (Servidor     │       │  (Este Notebook) │
│                 │       │  Legado → FHIR   │       │    FHIR)        │       │                  │
└─────────────────┘       └──────────────────┘       └─────────────────┘       └──────────────────┘
```

## Objetivos

1. ✅ Conectar ao Mini HAPI via API REST
2. ✅ Buscar pacientes (Patient)
3. ✅ Buscar observações (Observation)
4. ✅ Exibir dados em formato legível
5. ✅ Criar visualizações simples
6. ✅ Demonstrar o barramento funcionando ponta a ponta

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Configurações
FHIR_BASE_URL = os.getenv("FHIR_BASE_URL", "http://localhost:8000")
API_TOKEN = os.getenv("API_TOKEN", "troque-essa-chave")

HEADERS = {
    "Content-Type": "application/fhir+json",
    "Authorization": f"Bearer {API_TOKEN}"
}

# Configuração de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Bibliotecas importadas com sucesso!")
print(f"✓ Servidor FHIR: {FHIR_BASE_URL}")

In [ ]:
def get_all_patients():
    """Busca todos os pacientes do servidor FHIR"""
    try:
        response = requests.get(
            f"{FHIR_BASE_URL}/fhir/Patient",
            headers=HEADERS,
            params={"_count": 1000}
        )
        response.raise_for_status()
        bundle = response.json()
        
        patients = []
        if bundle.get("entry"):
            patients = [entry["resource"] for entry in bundle["entry"]]
        
        return patients
    except Exception as e:
        print(f"Erro ao buscar pacientes: {e}")
        return []

def get_patient_observations(patient_id):
    """Busca observações de um paciente específico"""
    try:
        response = requests.get(
            f"{FHIR_BASE_URL}/fhir/Observation",
            headers=HEADERS,
            params={"patient": patient_id, "_count": 1000}
        )
        response.raise_for_status()
        bundle = response.json()
        
        observations = []
        if bundle.get("entry"):
            observations = [entry["resource"] for entry in bundle["entry"]]
        
        return observations
    except Exception as e:
        print(f"Erro ao buscar observações: {e}")
        return []

def format_patient_info(patient):
    """Formata informações do paciente de forma legível"""
    name = "N/A"
    if patient.get("name"):
        name_obj = patient["name"][0]
        given = " ".join(name_obj.get("given", []))
        family = name_obj.get("family", "")
        name = f"{given} {family}".strip()
    
    return {
        "ID": patient.get("id", "N/A"),
        "Nome": name,
        "Gênero": patient.get("gender", "N/A"),
        "Data Nascimento": patient.get("birthDate", "N/A"),
        "Ativo": patient.get("active", "N/A")
    }

def format_observation_info(observation):
    """Formata informações da observação"""
    code_text = "N/A"
    if observation.get("code"):
        code_text = observation["code"].get("text", "N/A")
    
    value = "N/A"
    if observation.get("valueQuantity"):
        vq = observation["valueQuantity"]
        value = f"{vq.get('value', 'N/A')} {vq.get('unit', '')}"
    
    return {
        "ID": observation.get("id", "N/A"),
        "Exame": code_text,
        "Valor": value,
        "Status": observation.get("status", "N/A"),
        "Data": observation.get("effectiveDateTime", "N/A")
    }

print("✓ Funções auxiliares definidas!")

In [ ]:
# Busca todos os pacientes
print("🔍 Buscando pacientes do Mini HAPI...")
patients = get_all_patients()

print(f"✓ {len(patients)} pacientes encontrados!\n")

# Converte para DataFrame
if patients:
    patients_data = [format_patient_info(p) for p in patients]
    df_patients = pd.DataFrame(patients_data)
    
    print("📊 Primeiros 10 pacientes:")
    display(df_patients.head(10))
    
    # Estatísticas
    print("\n📈 Estatísticas:")
    print(f"  Total de pacientes: {len(df_patients)}")
    print(f"  Por gênero:")
    print(df_patients['Gênero'].value_counts().to_string())
else:
    print("⚠️ Nenhum paciente encontrado. Execute o conector ETL primeiro!")

In [ ]:
if patients:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico 1: Distribuição por gênero
    gender_counts = df_patients['Gênero'].value_counts()
    axes[0].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Distribuição de Pacientes por Gênero', fontsize=14, fontweight='bold')
    
    # Gráfico 2: Status ativo
    active_counts = df_patients['Ativo'].value_counts()
    axes[1].bar(active_counts.index.astype(str), active_counts.values, color=['green', 'orange'])
    axes[1].set_title('Status dos Pacientes', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Ativo')
    axes[1].set_ylabel('Quantidade')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Sem dados para visualizar")

In [ ]:
if patients:
    # Seleciona o primeiro paciente como exemplo
    sample_patient = patients[0]
    patient_id = sample_patient["id"]
    patient_info = format_patient_info(sample_patient)
    
    print(f"👤 Paciente selecionado: {patient_info['Nome']}")
    print(f"   ID: {patient_id}")
    print(f"   Gênero: {patient_info['Gênero']}")
    print(f"   Data Nascimento: {patient_info['Data Nascimento']}")
    print()
    
    # Busca observações deste paciente
    print("🔍 Buscando observações...")
    observations = get_patient_observations(patient_id)
    
    print(f"✓ {len(observations)} observações encontradas!\n")
    
    if observations:
        obs_data = [format_observation_info(obs) for obs in observations]
        df_observations = pd.DataFrame(obs_data)
        
        print("📊 Observações do paciente:")
        display(df_observations)
    else:
        print("⚠️ Nenhuma observação encontrada para este paciente")
else:
    print("⚠️ Execute as células anteriores primeiro!")

In [ ]:
if patients:
    print("🔍 Buscando todas as observações do sistema...")
    
    all_observations = []
    for patient in patients[:20]:  # Limita a 20 pacientes para performance
        obs = get_patient_observations(patient["id"])
        all_observations.extend(obs)
    
    print(f"✓ {len(all_observations)} observações encontradas!\n")
    
    if all_observations:
        # Análise de tipos de exames
        exam_types = {}
        for obs in all_observations:
            exam_name = obs.get("code", {}).get("text", "Desconhecido")
            exam_types[exam_name] = exam_types.get(exam_name, 0) + 1
        
        print("📊 Distribuição de Exames:")
        df_exams = pd.DataFrame(list(exam_types.items()), columns=['Exame', 'Quantidade'])
        df_exams = df_exams.sort_values('Quantidade', ascending=False)
        display(df_exams)
        
        # Visualização
        plt.figure(figsize=(12, 6))
        plt.barh(df_exams['Exame'], df_exams['Quantidade'], color='steelblue')
        plt.xlabel('Quantidade', fontsize=12)
        plt.title('Distribuição de Exames Realizados', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ Execute as células anteriores primeiro!")

## 7. Demonstração Ponta a Ponta

Este notebook demonstrou o **barramento FHIR funcionando completamente**:

### ✅ Fluxo Completo Verificado

1. **Sistema Legado (Simulador)** → Gerou dados em formato proprietário
2. **Conector ETL** → Transformou dados legados em recursos FHIR válidos
3. **Mini HAPI** → Armazenou e disponibilizou os recursos via API REST
4. **Visualização** → Consultou e exibiu os dados de forma compreensível

### 🎯 Conclusões

- ✅ A API FHIR está funcionando corretamente
- ✅ Os dados foram transformados com sucesso
- ✅ Pacientes e observações estão relacionados adequadamente
- ✅ O barramento está operacional de ponta a ponta

### 📈 Próximos Passos

1. Adicionar mais tipos de recursos FHIR (Practitioner, Encounter, etc.)
2. Implementar visualizações mais avançadas
3. Criar dashboards interativos
4. Integrar com Superset para análises aprofundadas

## 6. Análise de Todas as Observações

## 5. Buscar Observações de um Paciente Específico

## 4. Visualizar Distribuição de Pacientes

## 3. Buscar e Exibir Pacientes

## 2. Funções Auxiliares

## 1. Configuração e Importações